In [2]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import geopandas as gpd
from shapely.geometry import Point
from geodatasets import get_url
import re

import pandas as pd
import numpy as np

In [3]:
societies = pd.read_csv('../D-PLACE-dplace-cldf-908f302/cldf/societies.csv')

societies.head()

,ID,Name,Latitude,Longitude,Glottocode,Name_and_ID_in_source,xd_id,alt_names_by_society,main_focal_year,HRAF_name_ID,HRAF_ID,origLat,origLong,comment,glottocode_comment,region,type,Language_Level_Glottocodes,ISO639P3code,Contribution_ID
0,B72,!Kung,-20.00,21.18,juho1239,!Kung (B72),xd1,Kung Bushmen; !Kung (Was Nyae); Was Nyae,1950.0,San (FX10),FX10,-20.00,21.18,Original,NaN,Southern Africa,society,juho1239,NaN,dplace-dataset-binford
1,B296,Kuskowagmut,61.01,-161.55,kusk1241,Kuskowagmut (B296),xd1000,NaN,1850.0,NaN,NaN,61.01,-161.55,Original,NaN,Subarctic America,society,cent2127,NaN,dplace-dataset-binford
2,B322,Syilx,49.46,-119.63,sout2963,Okanagan (B322),xd1001,Okanagan,1860.0,NaN,NaN,49.46,-119.63,Original,NaN,Western Canada,society,okan1243,NaN,dplace-dataset-binford
3,B335,Round Lake Ojibwa,52.71,-90.62,seve1241,Round Lake Ojibwa (B335),xd1003,Weagamow; North Caribou Lake; Round Lake,1900.0,NaN,NaN,52.71,-90.62,Original,NaN,Eastern Canada,society,seve1240,NaN,dplace-dataset-binford
4,B339,North Albany Ojibwa,51.22,-83.10,wini1244,North Albany Ojibwa (B339),xd1004,North Albany,1850.0,NaN,NaN,51.22,-83.10,Original,NaN,Eastern Canada,society,seve1240,NaN,dplace-dataset-binford


In [4]:
#start gemini

# 2. Convert to a GeoDataFrame
# Replace 'lon' and 'lat' with your actual column names
geoData = gpd.GeoDataFrame(
    societies, 
    geometry=gpd.points_from_xy(societies['Longitude'], societies['Latitude']),
    crs="EPSG:4326"
)

In [5]:
# 3. Load the world map directly from the web
# Using the 110m resolution "admin_0_countries" map
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)

# 4. Filter to just what we need to keep it fast
world = world[['CONTINENT', 'geometry']]

In [6]:
# 5. Spatial Join
# 'predicate="within"' checks if the point is inside the continent polygon
result = gpd.sjoin(geoData, world, how="left", predicate="within")
# End Gemini

In [7]:
#just to see if there are any other "types"\
type_group = result.groupby('type').count()['ID']
type_group

type
languoid    4085
society     2599
Name: ID, dtype: int64

In [8]:
# I'm going to just focus on socieities
result_soc = result[result['type'] == 'society']
result_soc.shape

(2599, 23)

In [9]:
#Let's make an easier dataframe
SocDF = result_soc.drop(columns=['Glottocode', 'xd_id', 'HRAF_name_ID', 'comment', 'glottocode_comment', 'Language_Level_Glottocodes', 'ISO639P3code', 'Contribution_ID', 'type', 'HRAF_ID', 'origLat', 'origLong', 'geometry', 'index_right'])
#i removed 13 columns that I either didn't need/want or were redundant
SocDF.head()

,ID,Name,Latitude,Longitude,Name_and_ID_in_source,alt_names_by_society,main_focal_year,region,CONTINENT
0,B72,!Kung,-20.00,21.18,!Kung (B72),Kung Bushmen; !Kung (Was Nyae); Was Nyae,1950.0,Southern Africa,Africa
1,B296,Kuskowagmut,61.01,-161.55,Kuskowagmut (B296),NaN,1850.0,Subarctic America,North America
2,B322,Syilx,49.46,-119.63,Okanagan (B322),Okanagan,1860.0,Western Canada,North America
3,B335,Round Lake Ojibwa,52.71,-90.62,Round Lake Ojibwa (B335),Weagamow; North Caribou Lake; Round Lake,1900.0,Eastern Canada,North America
4,B339,North Albany Ojibwa,51.22,-83.10,North Albany Ojibwa (B339),North Albany,1850.0,Eastern Canada,North America


In [10]:
SocDF['main_focal_year'] = pd.to_numeric(SocDF['main_focal_year'])

In [11]:
americamap = {
    "North America": "Americas",
    "South America": "Americas",
    "Africa": "Africa",
    "Asia": "Asia",
    "Europe": "Europe",
    "Oceania": "Oceania"
}

SocDF['Continent2'] = SocDF['CONTINENT'].map(americamap)
SocDF.head()

,ID,Name,Latitude,Longitude,Name_and_ID_in_source,alt_names_by_society,main_focal_year,region,CONTINENT,Continent2
0,B72,!Kung,-20.00,21.18,!Kung (B72),Kung Bushmen; !Kung (Was Nyae); Was Nyae,1950.0,Southern Africa,Africa,Africa
1,B296,Kuskowagmut,61.01,-161.55,Kuskowagmut (B296),NaN,1850.0,Subarctic America,North America,Americas
2,B322,Syilx,49.46,-119.63,Okanagan (B322),Okanagan,1860.0,Western Canada,North America,Americas
3,B335,Round Lake Ojibwa,52.71,-90.62,Round Lake Ojibwa (B335),Weagamow; North Caribou Lake; Round Lake,1900.0,Eastern Canada,North America,Americas
4,B339,North Albany Ojibwa,51.22,-83.10,North Albany Ojibwa (B339),North Albany,1850.0,Eastern Canada,North America,Americas


In [12]:
continent = SocDF.groupby('Continent2').count()['ID']
continent

Continent2
Africa       709
Americas    1037
Asia         337
Europe       155
Oceania      155
Name: ID, dtype: int64

In [13]:
year_group = SocDF.groupby('main_focal_year').count()['ID']

year_group

main_focal_year
-6500.0    1
-2500.0    1
-2000.0    1
-1750.0    1
-1400.0    1
          ..
 1978.0    2
 1979.0    1
 1980.0    3
 1989.0    2
 1990.0    1
Name: ID, Length: 137, dtype: int64

In [14]:
worldepic = pd.read_csv('world-folk-epics-longlat.csv')

worldepic.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5


In [15]:
epicdata = pd.DataFrame(worldepic)

epicdata.head()

,Title,Description,Language,Tradition,Region,Year (Rough),Lat,Long
0,T'heydinn,"a Mauritanian epic ensemble, performed in Has...",Hassaniya,Mauritanian,African,1600.0,20.0,-12.0
1,Gassire's Lute,"an epic from the Soninke people, West Africa",Soninke,Soninke,African,400.0,15.0,-8.0
2,Bayajidda,a West African epic,Hausa,Hausa,African,900.0,12.5,7.5
3,Eri,a West African epic,Igbo,Igbo,African,900.0,6.0,7.0
4,Oduduwa,a West African epic,Yoruba,Yoruba,African,1000.0,7.5,4.5


In [16]:
region_group = epicdata.groupby('Region').count()['Title']

region_group

Region
African                                11
Albanian                                1
American                                2
Balto-Slavic                            8
Celtic                                  8
East and Central Asian                 14
Germanic                                8
Greek                                   3
Italic/Romance                          6
Northeast Caucasian                     1
South Asian (Indian Subcontinent)      18
Southeast Asian                        13
Southwest Asian (Arabian Peninsula)    21
Uralic                                  2
Name: Title, dtype: int64

In [17]:
regionmap1 = {
    "African": "Africa",
    "Albanian": "Europe",
    "American": "Americas",
    "Balto–Slavic": "Europe",
    "Celtic": "Europe",
    "East and Central Asian": "Asia",
    "Germanic": "Europe",
    "Greek": "Europe",
    "Italic/Romance": "Europe",
    "Northeast Caucasian": "Asia/Europe",
    "South Asian (Indian Subcontinent)": "Asia",
    "Southeast Asian": "Asia",
    "Southwest Asian (Arabian Peninsula)": "Asia",
    "Uralic": "Asia/Europe"
}

In [18]:
regionmap = {
    "African": "Africa",
    "Albanian": "Europe",
    "American": "Americas",
    "Balto–Slavic": "Europe",
    "Celtic": "Europe",
    "East and Central Asian": "Asia",
    "Germanic": "Europe",
    "Greek": "Europe",
    "Italic/Romance": "Europe",
    "Northeast Caucasian": "Asia",
    "South Asian (Indian Subcontinent)": "Asia",
    "Southeast Asian": "Asia",
    "Southwest Asian (Arabian Peninsula)": "Asia",
    "Uralic": "Europe"
}

In [19]:
epicdata['Continent'] = epicdata['Region'].map(regionmap)

In [20]:
epicscontinent = epicdata.groupby('Continent').count()["Title"]
epicscontinent

Continent
Africa      11
Americas     2
Asia        67
Europe      28
Name: Title, dtype: int64

In [21]:
#A traditions vs. epics comparison
fig3 = make_subplots(rows = 1, cols = 2, specs=[[{'type':'domain'}, {'type':'domain'}]])

fig3.add_trace(
    go.Pie(
    labels = continent.index,
    values=continent.values,
    title="Traditions by Continent"),
    row=1, col=1
    )
#I should write something short about what a tradition is, how it's essentially a culture or a larger idea of a tribe, a community

fig3.add_trace(
    go.Pie(
    labels = epicscontinent.index,
    values=epicscontinent.values,
    title="World Folk Epics by Continent"),
    row=1, col=2
    )

fig3.show()
#I chose to group North and South America so the comparison would be better
#I also edited the mapping so that Uralic is European and NE Caucasia is Asian so there wouldn't be a Asian/European category
#I'm thinking idk a stacked bar chart? I think there are better comparisons

In [22]:
# scatter timeline for societies
fig2 = go.Figure(data=[go.Scatter(x=year_group.index,
                    y=year_group.values,
                    mode='lines'
                    )])

fig2.show()

In [23]:
year_epics = epicdata.groupby('Year (Rough)').count()["Title"]
year_epics

Year (Rough)
-5500.0    1
-2500.0    1
-1500.0    1
-1292.0    1
-1200.0    1
          ..
 1838.0    1
 1861.0    1
 1881.0    1
 1900.0    1
 1939.0    1
Name: Title, Length: 71, dtype: int64

In [24]:
# scatter timeline for societies
fig2 = go.Figure(data=[go.Scatter(x=year_epics.index,
                    y=year_epics.values,
                    mode='lines'
                    )])

fig2.show()

In [25]:
fig9 = make_subplots(rows = 2, cols = 1, specs=[[{'type':'scatter'}], [{'type':'scatter'}]])

# scatter timeline for societies
fig9.add_trace(go.Scatter(x=year_epics.index,
                    y=year_epics.values,
                    name="Epics by Time",
                    mode='lines'
                    ), row=1, col=1
                    )

# scatter timeline for societies
fig9.add_trace(go.Scatter(x=year_group.index,
                    y=year_group.values,
                    name="Societies by Time",
                    mode='lines'
                    ), row=2, col=1
                    )

fig9.show()

#i don't think I'll include this visualization... it's mostly boring

In [26]:
epicdata = epicdata.dropna(subset=['Region'])
#I believe there must just be a single empty cell row because I got an error but everything has a region

regions = sorted(epicdata['Region'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color


layout = go.Layout(title="World Folk Epics by Tradition", legend_title="Region")
                #    , height=300, margin={"r":0,"t":0,"l":0,"b":0})

fig3 = go.Figure(layout=layout) 

# Loop through each unique Region to create separate traces
for r in epicdata['Region'].unique():
    epicdata_sub = epicdata[epicdata['Region'] == r]
    fig3.add_trace(go.Scattergeo(
                lat=epicdata_sub["Lat"],
                lon=epicdata_sub["Long"],
                text= epicdata_sub["Title"],
                customdata= epicdata_sub["Description"],
                name=r,
                mode='markers',
                marker=dict(
                    size=10,
                    opacity=0.6,
                    color=color_map[r])
    ))

fig3.update_geos(projection_type="natural earth")

fig3.update_traces(
    hovertemplate="<b>%{text}</b> "\
    "<br>%{customdata}",
    mode='markers',
    marker=dict(size=10, opacity=.5)
)
fig3.show()

# A comparison globally of popular storylines 

from https://www.folkloredatabase.com

using the database I did my best to find matching stories using these search terms:

**Eternal Tasks**
(repetitive OR continuous OR eternity OR eternal) (task OR punishment OR labor) OR (reward OR sacrifice)
3330 Narratives

**Warnings**
(warn* OR caution OR "do not") AND (greed or pride or hubris or hum* or lie*)

**Infidelity**
(lover OR lovers OR married OR wife) AND (infidelity OR cheating OR unfaithful)

**Magic**
magic* OR enchant* OR sorcer* OR witch* OR supernatural
9,330

**Shapeshifters** (this felt like the closest to magic without breaking the entire thing)
changeling OR shapeshifter OR "changes appearance" OR "steals faces" OR "stolen identity" OR "not who (he OR she OR they) seem"
93

In [27]:
etern = pd.read_csv('eternal-c.csv')
warn = pd.read_csv('warnings-c.csv')
infidel = pd.read_csv('infidelity-c.csv')
mag = pd.read_csv('magic-c.csv')
shape = pd.read_csv('changeling-c.csv')
# I cleaned all the csvs of the actual narrative text (which I understand is a frought decision)
# The csvs were all 10 or 20MB+ and without it I can actually load/see them in a way that's helpful
# the urls are still there, so I don't think I'm technically omitting important info

In [28]:
eternity = pd.DataFrame(etern)
warning  = pd.DataFrame(warn)
infidelity = pd.DataFrame(infidel)
magic = pd.DataFrame(mag)
shapeshift = pd.DataFrame(shape)

eternity.head()
# The biggest hurdles I'm anticipating are the random NaN values :/// ugg
# I also need to see If I just need to sort by Lat/Long or if the Loc/Tradition is narrow enough

,Motif Reference,Motif Title,Location / Tradition,Latitude,Longitude,source,berezkin,other_data
0,NaN,NaN,Italy,42.98200,12.76400,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN
1,NaN,NaN,Romania,45.82500,25.09400,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN
2,NaN,NaN,Europe,NaN,NaN,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN
3,NaN,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN
4,NaN,NaN,Wales,52.70169,-3.87158,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN


In [29]:
# List the columns you want to combine
all_locations = pd.concat([
    infidelity['Location / Tradition'],
    magic['Location / Tradition'],
    warning['Location / Tradition'],
    shapeshift['Location / Tradition'],
    eternity['Location / Tradition']
])

# Now get the total counts across all themes
loc_counts = all_locations.value_counts()
# with pd.option_context('display.max_rows', None):
#     print(loc_counts)

#Cool so that's way too many, back to geopandas

In [30]:
#Gemini did this from the loc_list.txt file I exported, I would have taken years honestly
#Also I still had to edit this, I noticed "Wales" was a dropped row originally, among others, sigh
region_map = {
    'African': [
        'African', 'Sub-Saharan', 'Nigeria', 'Kenya', 'Zulu', 'Bantu', 'Yoruba', 'Sudan',
        'Congo', 'Ethiopia', 'Somali', 'Mali', 'Ghana', 'Senegal', 'Tanzania', 'Madagascar'
    ],
    'Albanian': ['Albanian', 'Albania'],

    'American': [
        'American', 'North American', 'Native American', 'Inuit', 'Navajo', 'Cherokee',
        'Aztec', 'Maya', 'Inca', 'Kechua', 'Iroquois', 'Sioux', 'Apache', 'Brazil', 'Mexico', 'Canada', 'Jamaica', 'Antigua', "Zapotec", "Quiche"
    ],
    'Balto-Slavic': [
        'Russian Federation', 'Ukraine', 'Ukrainian', 'Belarus', 'Poland', 'Polish',
        'Czech', 'Slovak', 'Bulgaria', 'Bulgarian', 'Serbian', 'Croatian', 'Slovenian',
        'Latvia', 'Lithuania', 'Estonia', 'Slavic', 'Northern Ukrainians', 'Serbia', 'Moravia', 'Kashubians', 'Rusyns'
    ],
    'Celtic': [
    'Celtic', 
    'Wales', 'Welsh', 
    'Ireland', 'Irish', 'Gaelic', 
    'Scotland', 'Scottish', 
    'Brittany', 'Breton', 
    'Cornwall', 'Cornish', 
    'Isle of Man', 'Manx'],

    
    'East and Central Asian': [
        'Chinese', 'Japanese', 'Korean', 'Mongolian', 'Tibetan', 'Manchu', 'Taiwan', 'East Asian', 'Japan', 'China', 'Mongols'
    ],
    'Germanic': [
        'Germany', 'German', 'Iceland', 'Norway', 'Norwegian', 'Sweden', 'Swedish', 'Switzerland', 'Luxemborg',
        'Denmark', 'Danish', 'Dutch', 'Netherlands', 'Austria', 'English', 'Schleswig-Holstein',
        'Saxony', 'Bavaria', 'Lower Saxony', 'England', 'Scandinavia', 'Shetland Islands', 'Flanders', "Grimm",
        'Bartsch', 'Schoppner', 'Lyncker', "Kuhn", "Haas"
    ],
    'Greek': ['Greece', 'Greek', 'Hellenic', 'Cyprus'],

    'Italic/Romance': [
        'Italy', 'Italian', 'France', 'French', 'Spain', 'Spanish', 'Portugal', 'Portuguese',
        'Romania', 'Romance', 'Latin', 'Toscana', 'Sicily', 'Umbria', 'Marche', 'Lazio', "Perrault", "Basile"
    ],
    'Kartvelian / South Caucasian': [
        'Georgia', 'Georgian', 'Mingrelian', 'Svan', 'Laz', 'Kartvelian'
    ],
    'Levantine & North African': [
        'Levant', 'Syria', 'Lebanon', 'Palestine', 'Jordan', 'Egypt', 'Morocco', 'Tunisia',
        'Algeria', 'Libya', 'Maghreb', 'Berber'
    ],
    'Northeast Caucasian': [
        'Chechen', 'Ingush', 'Dagestan', 'Avar', 'Lezgin', 'Circassian', 'Armenians'
    ],
    'Oceanian / Pacific Islander': [
        'Oceania', 'Polynesia', 'Melanesia', 'Micronesia', 'Hawaii', 'Maori', 'Fiji',
        'Samoa', 'Tonga', 'Tuamotu', 'Gilbert Islands', 'Solomons'
    ],
    'South Asian': [
        'Indian', 'Vedic', 'Brahman', 'Hindu', 'Pakistan', 'Bengali', 'Tamil', 'Punjabi',
        'Sri Lanka', 'Nepal', 'Bangladesh', 'Panchtantra', 'Jatakas', 'Indian Literary'
    ],
    'Southeast Asian': [
        'Vietnam', 'Thailand', 'Thai', 'Indonesia', 'Malay', 'Philippines', 'Burma',
        'Myanmar', 'Cambodia', 'Laos', 'Singapore'
    ],
    'Southwest Asian': [
        'Arabian', 'Saudi', 'Yemen', 'Oman', 'Qatar', 'Kuwait', 'Emirates', 'Bedouin', 'Bedouins'
    ],
    'Turkic': [
        'Turkish', 'Turkey', 'Azerbaijan', 'Kazakh', 'Kirghiz', 'Uzbek', 'Turkmen',
        'Tatar', 'Bashkir', 'Uygur', 'Yakut', 'Tuva'
    ],
    'Uralic': [
        'Finnish', 'Finland', 'Hungarian', 'Hungary', 'Estonian', 'Sami', 'Uralic', 'Nenets'
    ],
    'West Asian / Indo-Iranian': [
        'Persian', 'Iran', 'Tajik', 'Kurdish', 'Baluchi', 'Pashto', 'Afghan', 'Persia', 'Armenia', 'Kurds',
        "Hartland", "Temme"
    ]
}

In [31]:
def process_folklore_data(filepath):
    # Load the data
    df = pd.read_csv(filepath)
    
    # Clean the column name (fixing the space issue: 'Location / Tradition')
    target_col = 'Location / Tradition'
    
    # 2. Assignment Logic
    def assign_region(loc_string):
        if pd.isna(loc_string):
            return 'Unknown'
        for region, keywords in region_map.items():
            if any(key.lower() in str(loc_string).lower() for key in keywords):
                #i love the above line! if it fits one word, it puts the region in, saves tons!
                return region
        return 'Other / Unclassified'

    # Create the new column
    df['Region_Group'] = df[target_col].apply(assign_region)
    
    # 3. Convert to GeoDataFrame (using your existing Lat/Long)
    # Removing rows without coordinates for cleaner mapping
    df = df.dropna(subset=['Latitude', 'Longitude'])
    geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
    gdf = gpd.GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)
    
    return gdf

# Apply it to your magic file
eternity = process_folklore_data('eternal-c.csv')
warning  = process_folklore_data('warnings-c.csv')
infidelity = process_folklore_data('infidelity-c.csv')
magic = process_folklore_data('magic-c.csv')
shapeshift = process_folklore_data('changeling-c.csv')
eternity.tail()
#great let's get these on a map

,Motif Reference,Motif Title,Location / Tradition,Latitude,Longitude,source,berezkin,other_data,Region_Group,geometry
3078,Ṛbhus - Book: 3 Hymn: 60,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3079,Ṛbhus - Book: 4 Hymn: 33,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3080,Ṛbhus - Book: 4 Hymn: 34,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3081,Ṛbhus - Book: 4 Hymn: 37,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)
3082,Ṛtu - Book: 1 Hymn: 15,NaN,"Indian Literary Tradition (Vedic, Brahman, Pur...",25.56227,83.08594,https://www.folkloredatabase.com/db_index.php?...,NaN,NaN,American,POINT (83.08594 25.56227)


In [32]:
regions = sorted(magic['Region_Group'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

layout = go.Layout(title="Magic", legend_title="Region")
                #    , height=300, margin={"r":0,"t":0,"l":0,"b":0})

fig7 = go.Figure(layout=layout) 

# Loop through each unique Region to create separate traces
for r in magic['Region_Group'].unique():
    magic_sub = magic[magic['Region_Group'] == r]
    fig7.add_trace(go.Scattergeo(
                lat=magic_sub["Latitude"],
                lon=magic_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=10,
                    opacity=0.6,
                    color=color_map[r])
    ))

fig7.update_geos(projection_type="natural earth")

fig7.show()

In [33]:
def spatial_classification(row):
    # Only fix the ones that are currently unclassified
    if row['Region_Group'] != 'Other/Unclassified':
        return row['Region_Group']
    
    lat, lon = row['Latitude'], row['Longitude']
    
    # Simple Polygon Logic (Approximate Bounding Boxes)
    if 35 <= lat <= 72 and -25 <= lon <= 45:
        return 'Europe (Spatial)'
    if -35 <= lat <= 35 and -20 <= lon <= 50:
        return 'Africa (Spatial)'
    if 15 <= lat <= 75 and -170 <= lon <= -50:
        return 'Americas (Spatial)'
    if -10 <= lat <= 80 and 50 <= lon <= 180:
        return 'Asia & Oceania (Spatial)'
        
    return 'Other/Still Undefined'

# Apply this to your 'magic' dataframe
magic['Region_Group'] = magic.apply(spatial_classification, axis=1)
#WHY IS NOTHING HAPPENING ;_; ugh

with pd.option_context('display.max_rows', None):
    print(magic['Region_Group'])

0                     Italic/Romance
1                     Italic/Romance
2                           Germanic
4                     Italic/Romance
5                           Germanic
6                     Italic/Romance
7                             Celtic
8                             Uralic
9                     Italic/Romance
10              Other / Unclassified
11                          Germanic
12                          Germanic
13                          Germanic
15                       South Asian
16                          Germanic
17                            Uralic
18                            Celtic
19                          Germanic
20                            Celtic
21                          Germanic
22                          Germanic
23                            Uralic
25                            Celtic
26                          Germanic
27                          Germanic
28                          Germanic
29                            Celtic
3

In [34]:
regions = sorted(magic['Region_Group'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

layout = go.Layout(title="Magic", legend_title="Region")

fig7 = go.Figure(layout=layout) 

# Loop through each unique Region to create separate traces
for r in magic['Region_Group'].unique():
    magic_sub = magic[magic['Region_Group'] == r]
    fig7.add_trace(go.Scattergeo(
                lat=magic_sub["Latitude"],
                lon=magic_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=10,
                    opacity=0.6,
                    color=color_map[r])
    ))

fig7.update_geos(projection_type="natural earth")

fig7.show()

In [35]:
fig8 = make_subplots(rows = 2, cols = 3, 
                    column_widths=[0.33, 0.33, 0.33], # Adjusted to match 3 columns
    row_heights=[0.5, 0.5],
    specs=[[{'type':'scattergeo'}, {'type':'scattergeo'}, None ],
           [{'type':'scattergeo'}, {'type':'scattergeo'}, {'type':'scattergeo'}]],
           subplot_titles=("Eternal Tasks", "Moral Heeding/Warning", "Infidelity in Relationships", "Magic/Soorcery", "Shapeshifting/Changelings"))

regions = sorted(magic['Region_Group'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

fig8.update_layout(title="Folklore Motifs by Category", legend_title="Region", showlegend=False)

# Loop through each unique Region to create separate traces
for r in eternity['Region_Group'].unique():
    eternity_sub = eternity[eternity['Region_Group'] == r]
    #eternity
    fig8.add_trace(go.Scattergeo(
                lat=eternity_sub["Latitude"],
                lon=eternity_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=1)
    
    #warnings
for r in warning['Region_Group'].unique():
    warning_sub = warning[warning['Region_Group'] == r]
    #eternity
    fig8.add_trace(go.Scattergeo(
                lat=warning_sub["Latitude"],
                lon=warning_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=2)

    #infidelity
for r in infidelity['Region_Group'].unique():
    infidelity_sub = infidelity[infidelity['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=infidelity_sub["Latitude"],
                lon=infidelity_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=1)

    #magic
for r in magic['Region_Group'].unique():
    magic_sub = magic[magic['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=magic_sub["Latitude"],
                lon=magic_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=2)
    
    #shapeshifting
for r in shapeshift['Region_Group'].unique():
    shapeshift_sub = shapeshift[shapeshift['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=shapeshift_sub["Latitude"],
                lon=shapeshift_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=3)

fig8.update_geos(projection_type="natural earth")

fig8.show()

In [36]:
fig8 = make_subplots(rows = 2, cols = 2, 
                    column_widths=[0.5, 0.5], # Adjusted to match 3 columns
    row_heights=[0.5, 0.5],
    specs=[[{'type':'scattergeo'}, {'type':'scattergeo'}],
           [{'type':'scattergeo'}, {'type':'scattergeo'}]],
           subplot_titles=("Eternal Tasks", "Moral Heeding/Warning", "Infidelity in Relationships", "Magic/Soorcery"))

regions = sorted(magic['Region_Group'].unique())

colors = px.colors.qualitative.Dark24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}
#I'm doing this bit above in order to have each region maintain a unique color

fig8.update_layout(title="Folklore Motifs by Category", legend_title="Region", showlegend=False)

# Loop through each unique Region to create separate traces
for r in eternity['Region_Group'].unique():
    eternity_sub = eternity[eternity['Region_Group'] == r]
    #eternity
    fig8.add_trace(go.Scattergeo(
                lat=eternity_sub["Latitude"],
                lon=eternity_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=1)
    
    #warnings
for r in warning['Region_Group'].unique():
    warning_sub = warning[warning['Region_Group'] == r]
    #eternity
    fig8.add_trace(go.Scattergeo(
                lat=warning_sub["Latitude"],
                lon=warning_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=1, col=2)

    #infidelity
for r in infidelity['Region_Group'].unique():
    infidelity_sub = infidelity[infidelity['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=infidelity_sub["Latitude"],
                lon=infidelity_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=1)

    #magic
for r in magic['Region_Group'].unique():
    magic_sub = magic[magic['Region_Group'] == r]
    fig8.add_trace(go.Scattergeo(
                lat=magic_sub["Latitude"],
                lon=magic_sub["Longitude"],
                name=r,
                mode='markers',
                marker=dict(
                    size=7,
                    opacity=0.3,
                    color=color_map[r])
    ),row=2, col=2)
    
    #i removed shapeshifting, doesn't aid the overall narrative

fig8.update_geos(projection_type="natural earth")

# 1. Calculate counts for each group
counts = {
    "eternity": len(eternity),
    "warning": len(warning),
    "infidelity": len(infidelity),
    "magic": len(magic)
}

# 2. Add annotations (x and y are from 0 to 1 across the whole figure)
# Adjust the 'y' values slightly if they overlap with your map borders
fig8.add_annotation(dict(x=0.22, y=0.6, text=f"{counts['eternity']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig8.add_annotation(dict(x=0.78, y=0.6, text=f"{counts['warning']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig8.add_annotation(dict(x=0.22, y=-0.06, text=f"{counts['infidelity']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))
fig8.add_annotation(dict(x=0.78, y=-0.06, text=f"{counts['magic']} data points", 
                         showarrow=False, xref="paper", yref="paper", font=dict(size=12)))

# 3. Increase bottom margin to make room for the bottom labels
fig8.update_layout(margin=dict(b=50))

fig8.show()

# Now I want to try something new... a timeline

I **think** it should be straightforward from the documentation https://plotly.com/python/animations/ I basically just need to nest my for loop that makes regions so then it can check for the time and then split the regions... if that makes sense. It does to me, so lets roll with it.

In [37]:
for year in epicdata["Year (Rough)"]:
    frame = {"data": []}

I changed my mind, that would have sucked and thinking about it I'm not even convinced it would show the data better.
https://plotly.com/python/mixed-subplots/

I like the idea of multiple subplots all interconnected to a storyline, lets make our globe with world epics then show maybe a timeline and a bar or pie chart to show disparity of data in different regions?

In [38]:
years = sorted(epicdata['Year (Rough)'].unique())

In [39]:
# I was suffering w the timeline aspect so I gave my code to Gemini
fig = go.Figure()

frame_data = []
# 1. Add Initial Traces (The starting point of the animation)
first_year = years[0]
for region in epicdata['Region'].unique():
    # Show only data from the very first available year
    mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= first_year)
    sub = epicdata[mask]
    
    fig.add_trace(go.Scattergeo(
        lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub["Description"],
        name=region,
        hovertemplate="<b>%{text}</b><br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
    ))

# 2. Create the Frames (One for each 'step' in time)
frames = []
for yr in years:
    frame_data = []
    for region in epicdata['Region'].unique():
        # Cumulative: Show everything up to the current slider year
        mask = (epicdata['Region'] == region) & (epicdata['Year (Rough)'] <= yr)
        sub = epicdata[mask]
        
        frame_data.append(go.Scattergeo(
            lat=sub["Lat"],
        lon=sub["Long"],
        text=sub["Title"],
        customdata=sub["Description"],
        name=region,
        hovertemplate="<b>%{text}</b> "\
            "<br>%{customdata}",
        mode='markers',
        marker=dict(size=10, opacity=.5)
        ))
    frames.append(go.Frame(data=frame_data, name=str(yr)))

fig.frames = frames

fig.update_layout(
    title="The Spread of Global Epics Over Time",
    showlegend=True,
    geo=dict(
        projection_type='natural earth',
        # showland=True,
        # landcolor="rgb(243, 243, 243)",
    ),
    updatemenus=[{
        "type": "buttons",
        "buttons": [{
            "label": "Play",
            "method": "animate",
            "args": [None, {"frame": {"duration": 500, "redraw": True}, "fromcurrent": True}]
        }]
    }],
    sliders=[{
        "active": 0,
        "steps": [{
            "label": str(yr),
            "method": "animate",
            "args": [[str(yr)], {
                "frame": {"duration": 100, "redraw": True}, 
                "fromcurrent": True,
                "transition": {"duration": 500},
                "mode": "immediate"}]
        } for yr in years]
    }]
)
#I think I'm going to get ride of the play button, and also try to fix the timeline scroll (bugs out if scrolled to the end)
fig.show()

In [40]:
# 1. Combine all unique regions from all datasets to ensure none are missed
all_regions = set(eternity['Region_Group'].unique()) | \
              set(warning['Region_Group'].unique()) | \
              set(betray['Region_Group'].unique()) | \
              set(magic['Region_Group'].unique())

regions = sorted(list(all_regions))

colors = px.colors.qualitative.Light24 # Or Light24, Bold, etc.

color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}

fig21 = go.Figure()

categories = ["Eternal Punishments", "Moral Warnings", "Betrayals", "Magic/Sorcery"]
datasets = [eternity, warning, betray, magic]
counts = [len(d) for d in datasets]

# We need to keep track of how many traces each category has to toggle visibility
trace_counts = []

# 2. Add traces for all categories to the same map
for data in datasets:
    unique_regions = data['Region_Group'].unique()
    trace_counts.append(len(unique_regions))
    
    for r in unique_regions:
        sub = data[data['Region_Group'] == r]
        fig21.add_trace(go.Scattergeo(
            lat=sub["Latitude"],
            lon=sub["Longitude"],
            name=r,
            mode='markers',
            marker=dict(size=7, opacity=0.4, color=color_map[r]),
            visible=False # Hide all initially
        ))

# Make the first category (Eternity) visible by default
for i in range(trace_counts[0]):
    fig21.data[i].visible = True

# 3. Create the visibility masks for the dropdown
buttons = []
start_idx = 0

for i, name in enumerate(categories):
    # Create a list of False for all traces
    visibility = [False] * len(fig21.data)
    # Set True only for the traces belonging to this category
    for j in range(start_idx, start_idx + trace_counts[i]):
        visibility[j] = True
    
    buttons.append(dict(
        label=name,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Folklore Motifs: {name} ({counts[i]} points)"}]
    ))
    start_idx += trace_counts[i]

# 4. Add the dropdown and clean up layout
fig21.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction="down",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.1,
        xanchor="left",
        y=1.1,
        yanchor="top"
    )],
    title=f"Folklore Motifs: {categories[0]} ({counts[0]} points)",
    showlegend=True # Legends now work globally per selection
)

fig21.update_geos(
    projection_type="natural earth",
    showcoastlines=False,
    showland=True, landcolor="DarkSeaGreen",
    showocean=True, oceancolor="Azure"
)

fig21.show()

NameError: name 'betray' is not defined